# Bronze Layer

### Import Spark types and functions used during raw dimension ingestion.

In [0]:
# Import the dependency used by the lines below.
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

### Load project-level config and define a small guard for empty source files.

In [0]:
# Load the catalog name from Spark config, or use the default project catalog.
catalog_name = spark.conf.get("training_0002_ecommerce.catalog_name", "training_0002_ecommerce")
source_base_path = spark.conf.get("training_0002_ecommerce.source_base_path", f"/Volumes/{catalog_name}/source_data/raw_data")

# Fail fast when an input file path exists but produces no records.
# Define a helper that stops the notebook when a required dataset is empty.
def validate_non_empty(df, dataset_name):
    # Count the rows so the dataset can be validated before writing it out.
    row_count = df.count()
    # Check whether the validation condition is met before continuing.
    if row_count == 0:
        # Stop execution with a clear error message when the validation fails.
        raise ValueError(f"{dataset_name} is empty. Check the configured source path before writing Bronze tables.")
    # Print a small status message so the notebook run is easier to follow.
    print(f"Validated {dataset_name}: {row_count} rows")

## Brands

### Read raw brand data, attach ingestion metadata, and preview the dataset.

In [0]:
# Define schema for the data file
brand_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
])

In [0]:
# Point Spark to the raw input file location for this dataset.
raw_data_path = f"{source_base_path}/brands.csv"

# Read the raw file into a DataFrame using the schema defined above.
df = spark.read.option('header', "true").option("delimeter", ",").schema(brand_schema).csv(raw_data_path)

# add metadata columns
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

validate_non_empty(df, "brands")
# Show a sample of the result so it can be visually checked.
display(df.limit(5))       

### Persist raw brand data into the Bronze layer as a Delta table.

In [0]:
# Write the current DataFrame to a Delta table in the target layer.
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

## Category

### Read raw category data, add metadata columns, and write Bronze output.

In [0]:
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])

# Load data using the schema defined
# Point Spark to the raw input file location for this dataset.
raw_data_path = f"{source_base_path}/category.csv"

# Read the raw file into a DataFrame using the schema defined above.
df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(category_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))

validate_non_empty(df_raw, "category")

# Write raw_data data to the Bronze layer (catalog: training_0002_ecommerce, schema: bronze, table: brz_category)
# Write the current DataFrame to a Delta table in the target layer.
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_category")               

## Products

### Read raw product data, apply the explicit schema, and capture file metadata.

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

# Load data using the schema defined
# Point Spark to the raw input file location for this dataset.
raw_data_path = f"{source_base_path}/products.csv"

# Read the raw file into a DataFrame using the schema defined above.
df = spark.read.option("header", "true").option("delimiter", ",").schema(products_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

validate_non_empty(df, "products")

# Write raw_data data to the Bronze layer (catalog: training_0002_ecommerce, schema: bronze, table: brz_products)
# Write the current DataFrame to a Delta table in the target layer.
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_products")    

## Customers

### Read raw customer data, add metadata, and write Bronze output.

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

# Load data using the schema defined
# Point Spark to the raw input file location for this dataset.
raw_data_path = f"{source_base_path}/customers.csv"

# Read the raw file into a DataFrame using the schema defined above.
df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(customers_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

validate_non_empty(df_raw, "customers")

# Write raw_data data to the Bronze layer (catalog: training_0002_ecommerce, schema: bronze, table: brz_customers)
# Write the current DataFrame to a Delta table in the target layer.
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")      

## Date

### Read raw calendar data, add ingestion metadata, and write Bronze output.

In [0]:

# Define schema for the data file
date_schema = StructType([
    StructField("date", StringType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])

# Load data using the schema defined
# Point Spark to the raw input file location for this dataset.
raw_data_path = f"{source_base_path}/date.csv" 

# Read the raw file into a DataFrame using the schema defined above.
df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(date_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_ingested_at", F.current_timestamp()) \
               .withColumn("_source_file", F.col("_metadata.file_path"))

validate_non_empty(df_raw, "calendar")

# Write raw_data data to the Bronze layer (catalog: training_0002_ecommerce, schema: bronze, table: brz_calendar) 
# Write the current DataFrame to a Delta table in the target layer.
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_calendar")               